In [ ]:
import json

dbutils.widgets.text("extractor_notebook_path", "")
dbutils.widgets.text("trigger", "")

extractor_notebook_path = dbutils.widgets.get("extractor_notebook_path")
trigger = json.loads(dbutils.widgets.get("trigger"))

In [ ]:
get_ipython().run_line_magic("run", extractor_notebook_path)

In [ ]:
data = ingest_data(**trigger["func_params"])

In [ ]:
from pyspark.sql.functions import current_timestamp

sink_table = trigger["sink_table"]
mode = trigger["mode"]
catalog, schema, table = sink_table.split(".")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {sink_table} (
        etl_ts TIMESTAMP
    )
    USING DELTA
""")

data = data.withColumn("etl_ts", current_timestamp())

In [ ]:
if mode == "overwrite":
    data.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable(sink_table)
elif mode == "append":
    data.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(sink_table)
elif mode == "merge":
    spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")
    merge_columns = trigger["merge_columns"]
    data.createOrReplaceTempView("worker_staging")
    join_condition = " AND ".join(f"target.{c} = source.{c}" for c in merge_columns)
    spark.sql(f"""
        MERGE INTO {sink_table} AS target
        USING worker_staging AS source
        ON {join_condition}
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)

In [ ]:
import json

trigger["func_params"] = update_config(data, trigger["func_params"])

dbutils.notebook.exit(json.dumps(trigger))